# Find the Promos

## 1. Configuration

Change the parameters below. `MODELS` supports one model or a comparison across several. The default runs Gemini 3.7 on all 50 episodes using 10-episode windows, stride 5 and up to 8 candidates per response. Medium thinking matches the successful trial.


In [23]:
from pathlib import Path

INPUT_DIR = Path("I Married a Broken Soldier. He was a Duke")
OUTPUT_DIR = Path("outputs/candidates")
BUDGET_PATH = Path("outputs/request_usage.json")
DEDUP_OUTPUT_DIR = Path("outputs/deduplicated")

# Input windows
WINDOW_SIZE = 10
STRIDE = 5

# Model and output
MODELS = ["gemini-3.7-flash"]
FALLBACK_MODEL = None
MAX_CANDIDATES = 8
THINKING_LEVEL = "medium"
MAX_OUTPUT_TOKENS = 65536

# API access and timing
DAILY_REQUESTS_PER_KEY = 5
TIMEOUT_SECONDS = 600
RETRIES = 2
RETRY_DELAY_SECONDS = 30
REQUEST_DELAY_SECONDS = 30

# Deduplication
DEDUP_OVERLAP_THRESHOLD = 0.8

# Judge and ranking
JUDGE_OUTPUT_DIR = Path("outputs/ranking")
JUDGE_INPUT_PATH = None  # Optional path to a saved deduplicated JSON report.
JUDGE_MODEL = "gemini-3.7-flash"
JUDGE_FALLBACK_MODEL = None
JUDGE_BATCH_SIZE = 8
JUDGE_THINKING_LEVEL = "medium"
JUDGE_MAX_OUTPUT_TOKENS = 65536
JUDGE_TOP_COMPARE_COUNT = 10
JUDGE_MIN_CORE_SCORE = 2
JUDGE_FORCE = False
JUDGE_PREVIEW_COUNT = 5
JUDGE_WEIGHTS = {
    "hook": 0.30,
    "cliffhanger": 0.30,
    "clarity": 0.25,
    "escalation": 0.15,
}

# Gemini keys are discovered automatically from .env and the environment.


## 2. Data preprocessing

Load the pack and create overlapping windows. Source IDs, original text and episode-relative timestamps are preserved. Warnings flag source timing inconsistencies.


In [24]:
import importlib
import scripts.preprocessing as preprocessing

importlib.reload(preprocessing)
from scripts.preprocessing import ScreenplayPreprocessor

preprocessor = ScreenplayPreprocessor(INPUT_DIR)
episodes = preprocessor.episodes
full_screenplay = preprocessor.full_text()
windows = preprocessor.create_windows(WINDOW_SIZE, STRIDE)

print(f"Episodes: {len(episodes)} | Windows: {len(windows)}")
print(f"Source entries: {sum(len(episode.entries) for episode in episodes)}")
for episode in episodes:
    for warning in episode.warnings:
        print("Source warning:", warning)

for window in windows:
    print(f"{window.window_id}: episodes {window.episode_numbers[0]}–"
          f"{window.episode_numbers[-1]} | {window.character_count:,} characters")


Episodes: 50 | Windows: 9
Source entries: 3727
Source warning: E022-B0023: entry extends outside scene 1
W001: episodes 1–10 | 105,546 characters
W002: episodes 6–15 | 91,135 characters
W003: episodes 11–20 | 92,214 characters
W004: episodes 16–25 | 100,704 characters
W005: episodes 21–30 | 103,410 characters
W006: episodes 26–35 | 111,364 characters
W007: episodes 31–40 | 110,270 characters
W008: episodes 36–45 | 91,357 characters
W009: episodes 41–50 | 83,404 characters


### Preview model input

Character counts are not token counts. Loglines and continuity notes provide context only; selections must reference screenplay source IDs.


In [25]:
print(windows[0].text[:1500])


Episode 1
CONTEXT ONLY (not selectable footage):
Logline: Lady Juliana arrives at the cathedral in her late mother's bridal gown, only to suffer the ultimate betrayal when her fiancé Gerald abandons her at the altar for her cruel sister Clarissa with her family's full blessing.
Continuity notes: First episode of the series. Establishes Juliana's humiliation and abandonment by her betrothed, Gerald, and her noble family in favor of her sister, Clarissa.
SCREENPLAY (timestamps are episode-relative MM:SS.fraction):

Scene 1: INT. CATHEDRAL – DAY [00:00.00–02:18.00]
[E001-B0001 | 00:00.00–00:04.80] CAMERA: Symmetrical framing on the heavy church doors as they slowly part, pushing in slightly as Juliana stands framed in the light, dressed in an elegant white silk gown.
[E001-B0002 | 00:01.00–00:04.80] ACTION: Juliana pauses at the threshold, her expression bright with joyful anticipation, wearing her mother's vintage gown and an oval pendant necklace.
[E001-B0003 | 00:04.80–00:06.80] CAMERA

## 3. Generate candidates across the full pack

Run each model on every window. Keys belong to separate projects; use the least-used key first and reserve each attempt against its daily budget, including retries and failures. Usage is saved in `BUDGET_PATH` and resets at midnight Pacific. Cached responses cost no requests. This local count cannot see earlier requests or usage outside this notebook; API quota errors still move to another key.

Successful responses and progress are saved. Rerun to reuse successes and retry failed windows. Exhausted daily budgets stop the run with progress preserved. Updating the prompt creates new requests rather than reusing old-prompt results.


In [ ]:
import scripts.promo_generator as promo_generator

importlib.reload(promo_generator)
from scripts.promo_generator import GenerationConfig, PromoGenerator, display_candidates

reports = {}
for model in MODELS:
    config = GenerationConfig(
        model=model,
        fallback_model=FALLBACK_MODEL,
        max_candidates=MAX_CANDIDATES,
        thinking_level=THINKING_LEVEL,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        timeout_seconds=TIMEOUT_SECONDS,
        retries=RETRIES,
        retry_delay_seconds=RETRY_DELAY_SECONDS,
        daily_requests_per_key=DAILY_REQUESTS_PER_KEY,
    )
    generator = PromoGenerator(
        config, output_dir=OUTPUT_DIR, budget_path=BUDGET_PATH
    )
    reports[model] = generator.generate_windows(
        windows=windows,
        episodes=episodes,
        request_delay_seconds=REQUEST_DELAY_SECONDS,
    )


## 4. Review results

Counts include duplicates from overlapping windows; ranking and deduplication come later. Valid means source and timing checks passed, not that the promo is editorially strong.


In [26]:
for model, report in reports.items():
    successful = [item for item in report["windows"] if item["status"] == "success"]
    failed = [item for item in report["windows"] if item["status"] == "failed"]
    candidates = [
        candidate
        for item in successful
        for candidate in item["result"]["candidates"]
    ]
    valid_count = sum(candidate["valid"] for candidate in candidates)

    print(f"\n{model}")
    pending = report["total_windows"] - len(report["windows"])
    print(f"Windows: {len(successful)} succeeded, {len(failed)} failed, {pending} pending")
    print(f"Candidates: {len(candidates)} total, {valid_count} valid")
    print("Progress report:", report["report_path"])
    for item in failed:
        print(item["window_id"], item["error"])



gemini-3.7-flash
Windows: 9 succeeded, 0 failed, 0 pending
Candidates: 51 total, 51 valid
Progress report: outputs/candidates/batch-8ba5d660f8edecfb.json


### Inspect one window

Select a model and window index to read the candidates and their validation results.


In [27]:
REVIEW_MODEL = MODELS[0]
REVIEW_WINDOW_INDEX = 0

record = reports[REVIEW_MODEL]["windows"][REVIEW_WINDOW_INDEX]
if record["status"] == "success":
    display_candidates(record["result"])
else:
    print(record["error"])


Model: gemini-3.7-flash
Token usage: {'cache_tokens_details': None, 'cached_content_token_count': None, 'candidates_token_count': 916, 'candidates_tokens_details': None, 'prompt_token_count': 45498, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 45498}], 'thoughts_token_count': 17389, 'tool_use_prompt_token_count': None, 'tool_use_prompt_tokens_details': None, 'total_token_count': 63803, 'traffic_type': None}
Saved to: outputs/candidates/W001-ab8258892e6a768a

1. The Altar Proposal
Duration: 56.3s | Valid: True
  Episode 3: 01:24.00–02:20.30
Hook: At E003-B0070 (01:24.00), an armed royal guard charges the Bearded Stranger, who immediately blocks the strike and drops him to the stone floor with a devastating punch, prompting Gerald to panic at E003-B0074.
Cliffhanger: At E003-B0116 (02:20.50), the Bearded Stranger sinks onto one knee before the stunned congregation and says, 'Lady Juliana... accept my hand,' cutting away before she responds.
Context: Viewers can see that 

## 5. Deduplicate overlapping selections

Use episode timestamps, not titles or an LLM. Clips match when every corresponding segment overlaps by at least `DEDUP_OVERLAP_THRESHOLD` of the shorter segment. Episode and playback order must match. Contained cuts also match. The first occurrence represents each group; all alternative cuts and explanations remain in the saved report. These rules apply to any series in the same input format.


In [28]:
import scripts.deduplication as deduplication

importlib.reload(deduplication)
from scripts.deduplication import PromoDeduplicator

deduplicator = PromoDeduplicator(DEDUP_OVERLAP_THRESHOLD)
deduplicated = {}

for model, report in reports.items():
    result = deduplicator.deduplicate(report)
    path = deduplicator.save(result, DEDUP_OUTPUT_DIR)
    deduplicated[model] = result

    print(model, result["summary"])
    print("Saved to:", path)


gemini-3.7-flash {'input_candidates': 51, 'unique_candidates': 35, 'duplicates': 16, 'invalid_candidates': 0}
Saved to: outputs/deduplicated/deduplicated-53daf886cdfb023f.json


### Unique promos

`unique_promos` contains the representatives for later ranking. Grouping shared footage does not prove that different cuts are equally effective; alternatives remain available for editorial review. Raw generation outputs are preserved.


In [29]:
unique_promos = deduplicated[MODELS[0]]["candidates"]
for promo in unique_promos:
    print(promo["candidate_id"], promo["title"], f"{promo['duration_seconds']}s")


W001-C001 The Altar Proposal 56.3s
W001-C002 Defending the Mother's Legacy 59.3s
W001-C003 The Cottage Standoff 37.7s
W002-C001 The Planted Theft Accusation 41.25s
W002-C002 The Blackthorn Ultimatum 43.51s
W002-C003 Defending the Medicinal Garden 41.8s
W002-C005 Rumors of the Duke 29.0s
W003-C001 The Stolen Watch Accusation 36.8s
W003-C004 Alexander Disarms and Interrogates 35.3s
W003-C005 Trapped in the Cottage Inferno 43.5s
W003-C006 The Presumed-Dead Duke 46.5s
W004-C004 A Wasting of the Lungs 32.8s
W004-C005 The Healer's Calling 36.8s
W004-C006 The Arsenic Antidote 53.0s
W004-C007 A Secret Infiltration Revealed 43.4s
W005-C002 The Disguised Maid's Signet Ring 47.08s
W005-C003 The Ashbourne Crest and the Armed Intrusion 51.5s
W005-C004 Framed for Poisoning the Matriarch 42.7s
W005-C005 Identity Unmasked Before the Gallows 41.1s
W005-C006 The Aristocratic Witchcraft Conspiracy 43.0s
W006-C004 The Witchcraft Accusation 51.1s
W006-C005 The Witchcraft Conspiracy 57.9s
W006-C006 Face to 

## 6. Judge and rank

Score the actual selected footage, hiding generator titles and explanations. The judge uses anchored 0–4 scores for hook, cliffhanger, clarity and escalation, with source evidence. Code calculates weighted scores and selects the strongest cut in each overlap group. The judge then compares the top recommended promos for posting order; both the weighted score and final rank are retained.

The 51 cuts require 7 scoring batches at the default settings, plus one shortlist comparison if at least two promos meet the quality threshold. Calls share the existing key budgets and cache. Incomplete or invalid judging stops the run; rerunning reuses successful batches. These are editorial judgments, not measured ad performance.

Run the configuration and preprocessing cells first. To rank saved results without repeating generation, set `JUDGE_INPUT_PATH` to a deduplicated JSON file. Otherwise this section uses `deduplicated` from Section 5.


In [ ]:
import json
import importlib
import scripts.promo_generator as promo_generator

# Reload dependencies before importing the judge.
importlib.reload(promo_generator)

import scripts.promo_judge as promo_judge
importlib.reload(promo_judge)
from scripts.promo_generator import GenerationConfig
from scripts.promo_judge import JudgeConfig, PromoJudge, display_ranked_promos

if JUDGE_INPUT_PATH is not None:
    saved_dedup = json.loads(Path(JUDGE_INPUT_PATH).read_text())
    judge_inputs = {saved_dedup["model"]: saved_dedup}
else:
    judge_inputs = deduplicated

judge_api_config = GenerationConfig(
    model=JUDGE_MODEL,
    fallback_model=JUDGE_FALLBACK_MODEL,
    thinking_level=JUDGE_THINKING_LEVEL,
    max_output_tokens=JUDGE_MAX_OUTPUT_TOKENS,
    timeout_seconds=TIMEOUT_SECONDS,
    retries=RETRIES,
    retry_delay_seconds=RETRY_DELAY_SECONDS,
    daily_requests_per_key=DAILY_REQUESTS_PER_KEY,
)
judge_config = JudgeConfig(
    batch_size=JUDGE_BATCH_SIZE,
    top_compare_count=JUDGE_TOP_COMPARE_COUNT,
    minimum_core_score=JUDGE_MIN_CORE_SCORE,
    weights=JUDGE_WEIGHTS,
)
judge = PromoJudge(judge_api_config, judge_config, JUDGE_OUTPUT_DIR, BUDGET_PATH)

ranked_results = {}
for source_model, source_report in judge_inputs.items():
    ranked_results[source_model] = judge.rank(
        deduplicated=source_report,
        episodes=episodes,
        request_delay_seconds=REQUEST_DELAY_SECONDS,
        force=JUDGE_FORCE,
    )


### Ranked output

JSON contains all ranks, scores, evidence, alternative assessments and segment timestamps. A matching Markdown file presents the required episode, ordered segment times, hook, cliffhanger, explanation and posting rationale. Candidates below the minimum core score remain visible but are flagged for review. Review the top picks and weak examples manually before submission.


In [30]:
for result in ranked_results.values():
    display_ranked_promos(result, limit=JUDGE_PREVIEW_COUNT)


Summary: {'cuts_judged': 51, 'groups_ranked': 35, 'recommended': 27, 'alternative_cuts_selected': 2}
Saved to: outputs/ranking/ranked-8a2533abb9172b24.json

#1 The Altar Proposal | 100.0/100
Recommended: True
  Episode 3: 01:24.00–02:20.30
Hook: A bearded stranger knocks down an armed guard in a church as the frantic groom yells in panic.
Cliffhanger: Will Lady Juliana accept the mysterious stranger's shocking marriage proposal in front of the congregation?
Why it works: It executes a classic fantasy romance rescue beats-in-order, moving from hard-hitting physical action to an electrifying public proposal.
Post here because: Establishes the core romantic premise and central characters immediately through an instant physical knockout and a dramatic public marriage proposal.
Concerns: ['The sudden transition from physical brawl to earnest romantic devotion feels slightly melodramatic.']

#2 Alexander Disarms and Interrogates | 93.75/100
Recommended: True
  Episode 16: 00:14.30–00:49.60
H